# CAFA Notebook

This notebook is the only run interface for the project.

Every future implementation commit must add the corresponding runnable cells here, together with its code, tests, and validation logic.

## Load dependencies

In [ ]:
from datetime import date
from pathlib import Path
import sys
from pprint import pprint
from tempfile import TemporaryDirectory

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

## Load project modules

In [ ]:
from cafa import ProjectConfig
from cafa.io import (
    benchmark_output_dir,
    build_recreated_layout,
    read_fasta_records,
    read_ia_values,
    read_test_taxon_rows,
    read_train_taxonomy_rows,
    read_train_term_rows,
    write_sequences,
    write_test_taxon_rows,
    write_train_taxonomy,
    write_train_terms,
)
from cafa.sources import (
    ResearchRequiredError,
    download_and_validate_go_obo,
    download_source,
    resolve_annotation_source_chain,
    resolve_go_obo_snapshot,
    resolve_uniprot_sprot_snapshot,
    sha256_file,
)
from cafa.ontology import (
    ancestors_of,
    canonicalize_go_id,
    propagate_scores,
    propagate_terms,
    read_go_obo,
    subontology_terms,
    terms_of_interest,
)
from cafa.types import ProteinTaxonRecord, ProteinTermRecord, SequenceRecord, SourceSnapshot, TaxonRecord
from cafa.validation import (
    validate_go_obo,
    validate_ia_values,
    validate_sequence_mapping,
    validate_train_taxonomy,
    validate_train_terms,
)


In [ ]:
from cafa.train import extract_train_taxonomy_records


## Create project config

In [ ]:
config = ProjectConfig(
    go_release="2025-06-01",
    train_uniprot_release="2025_03",
    submission_deadline=date(2026, 1, 27),
    evaluation_time=date(2026, 3, 17),
    train_taxon_ids=(),
    test_taxon_ids=(),
    subontologies=("MF", "BP", "CC"),
    evidence_codes=("EXP", "IDA"),
    similarity_backend="diamond",
    validation_mode="canonical",
    project_root=PROJECT_ROOT,
    cache_dir=Path(".cache/cafa"),
    recreated_data_dir=Path("recreated_comp_data"),
    artifacts_dir=Path("artifacts"),
    results_dir=Path("results"),
)

## Download and validate pinned GO file

In [ ]:
go_reference_path = PROJECT_ROOT / "comp_data" / "Train" / "go-basic.obo"
downloaded_go_path = download_and_validate_go_obo(config, go_reference_path)
go_snapshot = resolve_go_obo_snapshot(config)
uniprot_snapshot = resolve_uniprot_sprot_snapshot(config)
source_demo = {
    "go_snapshot": {
        "url": go_snapshot.url,
        "local_path": str(go_snapshot.local_path),
        "downloaded_path": str(downloaded_go_path),
    },
    "uniprot_snapshot": {
        "url": uniprot_snapshot.url,
        "local_path": str(uniprot_snapshot.local_path),
    },
    "downloaded_go_sha256": sha256_file(downloaded_go_path),
}

try:
    resolve_annotation_source_chain(config)
except ResearchRequiredError as exc:
    source_demo["annotation_source_chain"] = str(exc)

pprint(source_demo)

## Read ontology file

In [ ]:
go_path = downloaded_go_path
ontology = read_go_obo(go_path)

summary = {
    "release": ontology.release,
    "num_terms": len(ontology.terms),
    "num_alt_ids": len(ontology.alt_id_to_term_id),
    "mf_terms": len(subontology_terms(ontology, "MF")),
    "bp_terms": len(subontology_terms(ontology, "BP")),
    "cc_terms": len(subontology_terms(ontology, "CC")),
}

print("=" * 30 + " Downloaded GO graph file summary " + "=" * 30)
pprint(summary)
print("=" * 100)
sample_term = "GO:0000002"
sample_scores = {sample_term: 0.75}
print(f"sample term and score: {sample_scores}")
demo = {
    "canonical_term": canonicalize_go_id(ontology, sample_term),
    "ancestors": sorted(ancestors_of(ontology, sample_term))[:10],
    "propagated_terms": sorted(propagate_terms(ontology, {sample_term}))[:10],
    "propagated_scores": dict(sorted(propagate_scores(ontology, sample_scores).items())[:10]),
    "mf_terms_of_interest_sample": terms_of_interest(ontology, "MF")[:10],
}
pprint(demo)


## Recreating train artifacts

In [ ]:
layout = build_recreated_layout(PROJECT_ROOT / "recreated_comp_data")
mf_benchmark_dir = benchmark_output_dir(layout, "MF")

train_taxonomy_path = write_train_taxonomy(
    (
        ProteinTaxonRecord(protein_id="Q9Z", taxon_id=9606),
        ProteinTaxonRecord(protein_id="A0A", taxon_id=10090),
    ),
    layout.train_dir / "train_taxonomy.tsv",
)

train_terms_path = write_train_terms(
    (
        ProteinTermRecord(protein_id="Q9Z", term_id="GO:0003674", aspect="F"),
        ProteinTermRecord(protein_id="A0A", term_id="GO:0008150", aspect="P"),
    ),
    layout.train_dir / "train_terms.tsv",
)

train_sequences_path = write_sequences(
    (
        SequenceRecord(
            protein_id="Q9Z",
            header="sp|Q9Z|EXAMPLE_Q9Z Example protein Q9Z",
            sequence="M" * 64,
        ),
        SequenceRecord(
            protein_id="A0A",
            header=">sp|A0A|EXAMPLE_A0A Example protein A0A",
            sequence="ACDEFGHIK",
        ),
    ),
    layout.train_dir / "train_sequences.fasta",
)

test_taxa_path = write_test_taxon_rows(
    (
        TaxonRecord(taxon_id=9606, species_name="Homo sapiens"),
        TaxonRecord(taxon_id=10090, species_name="Mus musculus"),
    ),
    layout.test_dir / "testsuperset-taxon-list.tsv",
)

writer_demo = {
    "layout": {
        "root": str(layout.root_dir),
        "mf_benchmark_dir": str(mf_benchmark_dir),
    },
    "train_taxonomy_preview": train_taxonomy_path.read_text(encoding="utf-8"),
    "train_terms_preview": train_terms_path.read_text(encoding="utf-8"),
    "train_sequences_preview": train_sequences_path.read_text(encoding="utf-8"),
    "test_taxa_preview": test_taxa_path.read_text(encoding="utf-8"),
}

pprint(writer_demo)

## Loading/Reading train artifacts

In [ ]:
reader_demo = {
    "train_taxonomy_rows": read_train_taxonomy_rows(train_taxonomy_path),
    "train_term_rows": read_train_term_rows(train_terms_path),
    "train_sequence_records": read_fasta_records(train_sequences_path),
    "test_taxon_rows": read_test_taxon_rows(test_taxa_path),
    "ia_sample": dict(list(read_ia_values(PROJECT_ROOT / "comp_data" / "IA.tsv").items())[:3]),
}

pprint(reader_demo)

## Validating train artifacts

## Extract and validate train taxonomy

In [ ]:
import gzip
import io
import tarfile

train_taxonomy_config = ProjectConfig(
    go_release=config.go_release,
    train_uniprot_release=config.train_uniprot_release,
    submission_deadline=config.submission_deadline,
    evaluation_time=config.evaluation_time,
    train_taxon_ids=(9606,),
    test_taxon_ids=config.test_taxon_ids,
    subontologies=config.subontologies,
    evidence_codes=config.evidence_codes,
    similarity_backend=config.similarity_backend,
    validation_mode=config.validation_mode,
    project_root=config.project_root,
    cache_dir=config.cache_dir,
    recreated_data_dir=config.recreated_data_dir,
    artifacts_dir=config.artifacts_dir,
    results_dir=config.results_dir,
)

synthetic_flatfile = """ID   HUMAN_ONE               Reviewed;         5 AA;\nAC   P12345; Q54321;\nOX   NCBI_TaxID=9606;\nSQ   SEQUENCE   5 AA;  555 MW;  ABCDEF CRC64;\n     MAAAA\n//\nID   MOUSE_ONE               Reviewed;         5 AA;\nAC   P67890;\nOX   NCBI_TaxID=10090;\nSQ   SEQUENCE   5 AA;  555 MW;  ABCDEF CRC64;\n     MBBBB\n//\n"""
synthetic_archive_path = layout.train_dir / "synthetic_uniprot_sprot-only2025_03.tar.gz"
compressed_payload = io.BytesIO()
with gzip.GzipFile(fileobj=compressed_payload, mode="wb") as gzip_handle:
    gzip_handle.write(synthetic_flatfile.encode("utf-8"))
with tarfile.open(synthetic_archive_path, "w:gz") as archive:
    member_bytes = compressed_payload.getvalue()
    member = tarfile.TarInfo("uniprot_sprot.dat.gz")
    member.size = len(member_bytes)
    archive.addfile(member, io.BytesIO(member_bytes))

synthetic_snapshot = SourceSnapshot(
    name="uniprot_sprot",
    url=synthetic_archive_path.resolve().as_uri(),
    local_path=synthetic_archive_path,
    description="synthetic Swiss-Prot archive for notebook extraction demo",
)
extracted_train_taxonomy_rows = extract_train_taxonomy_records(
    train_taxonomy_config,
    synthetic_snapshot,
)
extracted_train_taxonomy_path = write_train_taxonomy(
    extracted_train_taxonomy_rows,
    layout.train_dir / "train_taxonomy.tsv",
)
reference_train_taxonomy_path = layout.train_dir / "reference_train_taxonomy.tsv"
reference_train_taxonomy_path.write_text(
    "P12345\t9606\nP67890\t10090\n",
    encoding="utf-8",
)
train_taxonomy_report = validate_train_taxonomy(
    extracted_train_taxonomy_path,
    reference_train_taxonomy_path,
    train_taxonomy_config,
)

train_taxonomy_demo = {
    "archive_path": str(synthetic_archive_path),
    "row_count": len(extracted_train_taxonomy_rows),
    "rows": extracted_train_taxonomy_rows,
    "validation_passed": train_taxonomy_report.passed,
    "validation_message": train_taxonomy_report.message,
}

pprint(train_taxonomy_demo)

In [ ]:
validation_root = PROJECT_ROOT / "recreated_comp_data" / "validation_demo"
validation_root.mkdir(parents=True, exist_ok=True)

recreated_taxonomy_path = validation_root / "recreated_train_taxonomy.tsv"
reference_taxonomy_path = validation_root / "reference_train_taxonomy.tsv"
recreated_taxonomy_path.write_text("EntryID\ttaxon_id\nA0A\t10090\n", encoding="utf-8")
reference_taxonomy_path.write_text("A0A\t10090\nQ9Z\t9606\n", encoding="utf-8")

recreated_terms_path = validation_root / "train_terms.tsv"
reference_terms_path = validation_root / "reference_train_terms.tsv"
reference_terms_taxonomy_path = validation_root / "train_taxonomy.tsv"
recreated_terms_path.write_text("EntryID\tterm\taspect\nA0A\tGO:0008150\tP\n", encoding="utf-8")
reference_terms_path.write_text("EntryID\tterm\taspect\nA0A\tGO:0008150\tP\nQ9Z\tGO:0003674\tF\n", encoding="utf-8")
reference_terms_taxonomy_path.write_text("A0A\t10090\nQ9Z\t9606\n", encoding="utf-8")

recreated_sequence_path = validation_root / "recreated_train_sequences.fasta"
reference_sequence_path = validation_root / "reference_train_sequences.fasta"
recreated_sequence_path.write_text(">sp|A0A|EXAMPLE_A0A Example protein A0A\nACDEFGHIK\n", encoding="utf-8")
reference_sequence_path.write_text(">sp|A0A|EXAMPLE_A0A Example protein A0A\nACDEFGHIK\n", encoding="utf-8")

validation_config = ProjectConfig(
    go_release="2025-06-01",
    train_uniprot_release="2025_03",
    submission_deadline=date(2026, 1, 27),
    evaluation_time=date(2026, 3, 17),
    train_taxon_ids=(10090,),
    test_taxon_ids=(),
    subontologies=("BP",),
    evidence_codes=("EXP", "IDA"),
    similarity_backend="diamond",
    validation_mode="canonical",
    project_root=PROJECT_ROOT,
    cache_dir=Path(".cache/cafa"),
    recreated_data_dir=Path("recreated_comp_data"),
    artifacts_dir=Path("artifacts"),
    results_dir=Path("results"),
)

validation_demo = {
    "train_taxonomy_report": validate_train_taxonomy(
        recreated_taxonomy_path,
        reference_taxonomy_path,
        validation_config,
    ),
    "train_terms_report": validate_train_terms(
        recreated_terms_path,
        reference_terms_path,
        validation_config,
        ontology,
        taxonomy_rows=(ProteinTaxonRecord(protein_id="A0A", taxon_id=10090),),
    ),
    "sequence_report": validate_sequence_mapping(
        recreated_sequence_path,
        reference_sequence_path,
        artifact_name="Train/train_sequences.fasta",
    ),
}
pprint(validation_demo)

## Valdidating ontology file and IA file

In [ ]:
validation_extra_root = PROJECT_ROOT / "recreated_comp_data" / "validation_demo_extra"
validation_extra_root.mkdir(parents=True, exist_ok=True)

recreated_obo_path = validation_extra_root / "recreated_go.obo"
reference_obo_path = validation_extra_root / "reference_go.obo"
recreated_obo_path.write_text((PROJECT_ROOT / "comp_data" / "Train" / "go-basic.obo").read_text(encoding="utf-8"), encoding="utf-8")
reference_obo_path.write_text((PROJECT_ROOT / "comp_data" / "Train" / "go-basic.obo").read_text(encoding="utf-8"), encoding="utf-8")

recreated_ia_path = validation_extra_root / "recreated_IA.tsv"
reference_ia_path = validation_extra_root / "reference_IA.tsv"
recreated_ia_path.write_text("GO:0000001\t0.0\nGO:0000002\t2.8496657269155685\n", encoding="utf-8")
reference_ia_path.write_text("GO:0000001\t0.0\nGO:0000002\t2.8496657269155685\n", encoding="utf-8")

validation_extra_demo = {
    "go_report": validate_go_obo(recreated_obo_path, reference_obo_path),
    "ia_report": validate_ia_values(recreated_ia_path, reference_ia_path),
}

pprint(validation_extra_demo)
pprint("=============================")

with TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    recreated_path = root / "recreated_ia.tsv"
    reference_path = root / "reference_ia.tsv"
    recreated_path.write_text("GO:0000001\t1.5\n", encoding="utf-8")
    reference_path.write_text("GO:0000001\t1.0\nGO:0000002\t2.0\n", encoding="utf-8")

    report = validate_ia_values(
        recreated_path,
        reference_path,
        relative_tolerance=1e-12,
        absolute_tolerance=1e-12,
    )
    pprint(report)